First need to load the data in.

In [2]:
from pathlib import Path
import sys

from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "analyses" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loading import DEFAULT_CSV_PATH, URL, download_spf_data, save_csv

# --- Read directly from the web ---
df = download_spf_data(URL)

# --- Optional: inspect columns ---
print(df.columns)
print(df.head())

# --- Save as CSV ---
csv_path = save_csv(df, DEFAULT_CSV_PATH)

print(f"CSV saved as {csv_path}")


Index(['YEAR', 'QUARTER', 'ID', 'INDUSTRY', 'CPI1', 'CPI2', 'CPI3', 'CPI4',
       'CPI5', 'CPI6', 'CPIA', 'CPIB', 'CPIC'],
      dtype='str')
   YEAR  QUARTER  ID  INDUSTRY  CPI1  CPI2  CPI3  CPI4  CPI5  CPI6  CPIA  \
0  1968        4   1       NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   
1  1968        4   2       NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   
2  1968        4   3       NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   
3  1968        4   4       NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   
4  1968        4   5       NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   

   CPIB  CPIC  
0   NaN   NaN  
1   NaN   NaN  
2   NaN   NaN  
3   NaN   NaN  
4   NaN   NaN  
CSV saved as /home/clayt/Ensemble-Forecasting/data/SPF_Individual_CPI.csv


/home/clayt/miniconda3/envs/ensemble/lib/python3.11/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


## Exploratory Data Analysis

This section inspects missingness, coverage over time, and how consistently individual forecasters report.


In [3]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "analyses" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.cleaning import (
    average_forecasts_by_period,
    long_forecast_frame,
    missingness_by_horizon,
    prepare_spf_data,
)
from src.data.loading import DEFAULT_CSV_PATH, URL, load_or_download_csv
from src.data.plots import (
    analyze_optimal_reporting_windows,
    plot_average_forecasts_over_time,
    plot_dispersion_scatter,
    plot_missingness_bar,
    plot_reporting_forecasters_over_time,
    plot_reporting_waterfall,
    summarize_and_plot_reporting_consistency,
)

data_path = DEFAULT_CSV_PATH
df_raw = load_or_download_csv(data_path, URL)
df, forecast_cols = prepare_spf_data(df_raw)

df


ImportError: cannot import name 'realiz5_to_actual_inflation' from 'src.data.loading' (/home/clayt/Ensemble-Forecasting/src/data/loading.py)

In [ ]:
# Missingness by horizon
missing_rate = missingness_by_horizon(df, forecast_cols)
missing_rate


In [ ]:
plot_missingness_bar(missing_rate)


In [ ]:
# Count of forecasters reporting per quarter, by horizon
plot_reporting_forecasters_over_time(df, forecast_cols)


In [ ]:
# Consistency: fraction of periods in which each forecaster reports, by horizon
summarize_and_plot_reporting_consistency(df, forecast_cols, display_fn=display)


In [ ]:
# Find the optimal contiguous period window maximizing fully-reporting forecasters, by horizon
# Definition: a forecaster is 'fully reporting' if they reported in every period within the window.
# Constraint: minimum window length of 15 years (60 quarters).
# Optional constraint: require the window to include a specific date (set REQUIRED_DATE).

MIN_YEARS = 8
REQUIRED_DATE = None  # e.g., '2008-09-30' to require the window include 2008Q3

analyze_optimal_reporting_windows(
    df,
    forecast_cols,
    min_years=MIN_YEARS,
    required_date=REQUIRED_DATE,
)


In [ ]:
# Forecasts over time: average by horizon across forecasters (separate figures)
avg_by_period = average_forecasts_by_period(df, forecast_cols)
plot_average_forecasts_over_time(avg_by_period, forecast_cols)


In [ ]:
# Dispersion scatter: separate figures for each horizon
long = long_forecast_frame(df, forecast_cols)
plot_dispersion_scatter(long)


In [ ]:
# Waterfall-style plot: forecaster reporting over time (ID x period), per horizon
plot_reporting_waterfall(df, forecast_cols)


In [ ]:
# SPF + VIX experiment setup (Realiz5 actuals + uncertainty inputs)
import importlib
import pandas as pd
import src.data.cleaning as cln
cln = importlib.reload(cln)

from src.data.loading import (
    DEFAULT_SPF_CPI_REALIZ5_CSV_PATH,
    DEFAULT_VIX_CSV_PATH,
    SPF_CPI_REALIZ5_URL,
    VIX_CSV_URL,
    load_or_download_spf_realiz5_actuals,
    load_or_download_vix_data,
)
from src.data.cleaning import (
    alignment_diagnostics_spf_offsets,
    build_spf_cpi_panel_with_actuals,
    clean_spf_realiz5_actuals,
    find_optimal_reporting_window,
    prepare_vix_data,
    realiz5_to_actual_inflation,
)

realiz_raw = load_or_download_spf_realiz5_actuals(
    csv_path=DEFAULT_SPF_CPI_REALIZ5_CSV_PATH,
    url=SPF_CPI_REALIZ5_URL,
)
vix_raw = load_or_download_vix_data(
    csv_path=DEFAULT_VIX_CSV_PATH,
    url=VIX_CSV_URL,
)

cpi_actuals_level = clean_spf_realiz5_actuals(realiz_raw, realiz_col='Realiz5')
actual_inflation = realiz5_to_actual_inflation(
    cpi_actuals_level,
    assume_already_annualized=True,
)
actual_inflation = actual_inflation.rename(columns={'time': 'period'})
actual_inflation['period'] = pd.PeriodIndex(pd.to_datetime(actual_inflation['period']), freq='Q')

vix_daily, vix_quarterly = prepare_vix_data(vix_raw)

print('realiz_raw shape:', realiz_raw.shape)
print('cpi_actuals_level shape:', cpi_actuals_level.shape)
print('actual_inflation shape:', actual_inflation.shape)
print('vix_daily shape:', vix_daily.shape)
print('vix_quarterly shape:', vix_quarterly.shape)

display(cpi_actuals_level.head())
display(actual_inflation.head())
display(vix_quarterly.head())



In [ ]:
# Build SPF panel aligned to actuals and pick optimal windows for CPI1-3
import pandas as pd

HORIZONS = ['CPI1', 'CPI2', 'CPI3']
MIN_YEARS = 8

# Step 1: check alignment offsets explicitly (lower MAE / MSE and higher corr is better)
align_diag = alignment_diagnostics_spf_offsets(
    spf_clean=df,
    actual_inflation=actual_inflation,
    horizons=HORIZONS,
    offsets=[-2, -1, 0, 1],
)

print('Alignment diagnostics by offset and horizon:')
display(align_diag.sort_values(['horizon_name', 'offset']).reset_index(drop=True))

# Step 2: select a common offset by minimizing mean MAE across horizons
offset_scores = (
    align_diag.groupby('offset', as_index=False)
    .agg(mean_mae=('mae_consensus', 'mean'), mean_mse=('mse_consensus', 'mean'))
    .sort_values(['mean_mae', 'mean_mse'])
)
SELECTED_TARGET_OFFSET = int(offset_scores.iloc[0]['offset'])
print('Selected target offset:', SELECTED_TARGET_OFFSET)
display(offset_scores)

# Step 3: build panel with selected offset
panel_long = build_spf_cpi_panel_with_actuals(
    spf_clean=df,
    actual_inflation=actual_inflation,
    horizons=HORIZONS,
    target_offset=SELECTED_TARGET_OFFSET,
)

windows = {}
for h in HORIZONS:
    windows[h] = find_optimal_reporting_window(
        spf_clean=df,
        forecast_col=h,
        min_years=MIN_YEARS,
        required_date=None,
    )

windows_df = pd.DataFrame([
    {
        'horizon': h,
        'start_period': str(windows[h]['start_period']),
        'end_period': str(windows[h]['end_period']),
        'n_periods': windows[h]['n_periods'],
        'n_forecasters': windows[h]['n_ids'],
    }
    for h in HORIZONS
])

print('panel_long shape:', panel_long.shape)
display(windows_df)
display(panel_long.head())



In [ ]:
# Plot average SPF CPI forecasts vs cleaned actuals (aligned by target quarter)
import matplotlib.pyplot as plt

plot_panel = build_spf_cpi_panel_with_actuals(
    spf_clean=df,
    actual_inflation=actual_inflation,
    horizons=['CPI1', 'CPI2', 'CPI3'],
    target_offset=int(globals().get('SELECTED_TARGET_OFFSET', -1)),
)

plot_panel = plot_panel.copy()
plot_panel['target_time'] = plot_panel['target_period'].dt.to_timestamp('Q')

avg_fc = (
    plot_panel
    .groupby(['horizon_name', 'target_time'], as_index=False)
    .agg(avg_forecast=('forecast', 'mean'), actual=('actual', 'last'))
)

fig, axes = plt.subplots(3, 1, figsize=(12, 11), sharex=True)
for ax, h in zip(axes, ['CPI1', 'CPI2', 'CPI3']):
    d = avg_fc.loc[avg_fc['horizon_name'] == h].sort_values('target_time')
    ax.plot(d['target_time'], d['avg_forecast'], label=f'{h} avg SPF forecast', linewidth=1.8)
    ax.plot(d['target_time'], d['actual'], label='Actual annualized CPI', linewidth=1.8)
    ax.set_title(f'{h}: Average SPF Forecast vs Actual')
    ax.set_ylabel('Annualized %')
    ax.grid(alpha=0.25)
    ax.legend(loc='best')

axes[-1].set_xlabel('Target Quarter')
plt.tight_layout()
plt.show()




In [ ]:
# Run ensembling for CPI1-3 using PCA state (VIX + disagreement + error volatility)
import importlib
import numpy as np

import src.evaluation.spf_experiment as spfexp
spfexp = importlib.reload(spfexp)

from src.data.cleaning import build_spf_horizon_matrix
from src.evaluation import optuna_tuning as ot
aggregate_spf_results = spfexp.aggregate_spf_results
build_spf_state_components = spfexp.build_spf_state_components
evaluate_spf_horizon_matrix = spfexp.evaluate_spf_horizon_matrix
pca_state_from_components = spfexp.pca_state_from_components

LOSS_SECTIONS = ['mse', 'linex']
LINEX_A = 0.05
INCLUDE_RL = True
MIN_OBS = 24
MIN_FORECASTERS = 3
ERROR_VOL_WINDOW = 8

params_map = {k: dict(v) for k, v in ot.DEFAULT_METHOD_PARAMS.items()}
kappa_grid = np.array([0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 8.0], dtype=float)

detailed_rows_spf = []
diag_rows_spf = []
summary_rows_spf_h = []
state_frames = {}

for h in HORIZONS:
    w = windows[h]
    mat_obj = build_spf_horizon_matrix(
        panel_long=panel_long,
        horizon_name=h,
        start_period=w['start_period'],
        end_period=w['end_period'],
        forecaster_ids=w['ids'],
        min_obs=MIN_OBS,
        min_forecasters=MIN_FORECASTERS,
    )
    if mat_obj is None:
        print(f'SKIP {h}: no feasible matrix after window/ID filtering')
        continue

    components = build_spf_state_components(
        mat=mat_obj['mat'],
        vix_quarterly=vix_quarterly,
        error_vol_window=ERROR_VOL_WINDOW,
    )
    state = pca_state_from_components(
        components=components,
        expanding=True,
        min_train=12,
    )
    components['state_pca'] = state
    state_frames[h] = components

    rows_h, diag_h, summary_h = evaluate_spf_horizon_matrix(
        horizon_name=h,
        mat=mat_obj['mat'],
        y=mat_obj['y'],
        F=mat_obj['F'],
        state=state,
        loss_sections=LOSS_SECTIONS,
        linex_a=float(LINEX_A),
        include_rl=bool(INCLUDE_RL),
        params_map=params_map,
        kappa_grid=kappa_grid,
    )
    detailed_rows_spf.extend(rows_h)
    diag_rows_spf.extend(diag_h)
    summary_rows_spf_h.extend(summary_h)

summary_rows_spf_overall = aggregate_spf_results(detailed_rows_spf)

summary_h_df = pd.DataFrame(summary_rows_spf_h)
if not summary_h_df.empty:
    summary_h_df = summary_h_df.sort_values(['horizon_name', 'loss_section', 'objective_mean']).reset_index(drop=True)

summary_overall_df = pd.DataFrame(summary_rows_spf_overall)
if not summary_overall_df.empty:
    summary_overall_df = summary_overall_df.sort_values(['loss_section', 'objective_mean']).reset_index(drop=True)

print('SPF detailed rows:', len(detailed_rows_spf))
print('SPF diagnostic rows:', len(diag_rows_spf))
display(summary_h_df)
display(summary_overall_df)





In [ ]:
# Persist SPF experiment outputs
from src.evaluation.m3_macro_experiment import write_csv

OUT_DIR = PROJECT_ROOT / 'analyses' / 'results' / 'spf'
OUT_DIR.mkdir(parents=True, exist_ok=True)

write_csv(OUT_DIR / 'spf_cpi123_detailed.csv', detailed_rows_spf)
write_csv(OUT_DIR / 'spf_cpi123_policy_diagnostics.csv', diag_rows_spf)
write_csv(OUT_DIR / 'spf_cpi123_summary_by_horizon.csv', summary_rows_spf_h)
write_csv(OUT_DIR / 'spf_cpi123_summary_overall.csv', summary_rows_spf_overall)

for h, sdf in state_frames.items():
    sdf.to_csv(OUT_DIR / f'spf_state_components_{h}.csv', index=False)

print('Wrote outputs to', OUT_DIR)

